In [1]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [2]:
documents = [
    {
        "source": "Revenue Management Notes",
        "topic": "RevPAR",
        "question": "What is RevPAR?",
        "answer": "RevPAR means Revenue Per Available Room. It is calculated as room revenue divided by available rooms, or ADR multiplied by occupancy."
    },
    {
        "source": "Revenue Management Notes",
        "topic": "ADR",
        "question": "What is ADR?",
        "answer": "ADR means Average Daily Rate. It is calculated as room revenue divided by the number of rooms sold."
    },
    {
        "source": "Revenue Management Notes",
        "topic": "Occupancy",
        "question": "What is occupancy?",
        "answer": "Occupancy is the percentage of available rooms that are sold during a specific period."
    },
    {
        "source": "Revenue Management Notes",
        "topic": "Dynamic Pricing",
        "question": "What is dynamic pricing?",
        "answer": "Dynamic pricing is the practice of adjusting room rates based on demand, availability, seasonality, events, and competitor pricing."
    },
    {
        "source": "Revenue Management Notes",
        "topic": "Channel Management",
        "question": "What is channel management?",
        "answer": "Channel management is the process of managing hotel availability, rates, and inventory across OTAs, direct booking channels, GDS, and other distribution channels."
    }
]

In [3]:
def search(query):
    results = []

    for doc in documents:
        text = f'{doc["topic"]} {doc["question"]} {doc["answer"]}'.lower()
        score = 0

        for word in query.lower().split():
            if word in text:
                score += 1

        if score > 0:
            results.append((score, doc))

    results = sorted(results, reverse=True, key=lambda x: x[0])
    return [doc for score, doc in results[:3]]

In [4]:
search("How do I calculate RevPAR?")

[{'source': 'Revenue Management Notes',
  'topic': 'RevPAR',
  'question': 'What is RevPAR?',
  'answer': 'RevPAR means Revenue Per Available Room. It is calculated as room revenue divided by available rooms, or ADR multiplied by occupancy.'},
 {'source': 'Revenue Management Notes',
  'topic': 'ADR',
  'question': 'What is ADR?',
  'answer': 'ADR means Average Daily Rate. It is calculated as room revenue divided by the number of rooms sold.'},
 {'source': 'Revenue Management Notes',
  'topic': 'Occupancy',
  'question': 'What is occupancy?',
  'answer': 'Occupancy is the percentage of available rooms that are sold during a specific period.'}]

In [5]:
def build_prompt(query, search_results):
    context = ""

    for doc in search_results:
        context += f"""
Source: {doc["source"]}
Topic: {doc["topic"]}
Question: {doc["question"]}
Answer: {doc["answer"]}
"""

    prompt = f"""
You are Revenue AI Copilot, an assistant specialized in hotel revenue management.

Answer the QUESTION using only the CONTEXT.
If the answer is not in the CONTEXT, say "I don't know based on the available documents."

QUESTION:
{query}

CONTEXT:
{context}
"""
    return prompt

In [6]:
def llm(prompt, model="llama-3.1-8b-instant"):
    response = openai_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content

In [7]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    return answer

In [8]:
rag("How do I calculate RevPAR?")

'To calculate RevPAR (Revenue Per Available Room), I would use the formula as mentioned in the Context: \n\nRevPAR = ADR * Occupancy \n\nOr, alternatively, I would also use: \n\nRevPAR = Room Revenue / Available Rooms \n\nBoth formulas are mentioned in the Context.'

In [9]:
rag("What is dynamic pricing?")

'Dynamic pricing is the practice of adjusting room rates based on demand, availability, seasonality, events, and competitor pricing.'